In [ ]:
!pip install spacy
!pip install vaderSentiment

In [3]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import spacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Example sentences
sentences = [
    "I love programming.",
    "I hate bugs.",
    "Coding is fun.",
    "Debugging is frustrating.",
    "She enjoys solving complex problems.",
    "The movie was incredibly exciting but a bit long.",
]

# Step 1: Label words as S, V, O, or M
def label_word(word):
    doc = nlp(word)
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN", "PRON"]:
            return "S"  # Subject
        elif token.pos_ in ["VERB", "AUX"]:
            return "V"  # Verb
        elif token.pos_ in ["ADJ", "ADV", "DET", "ADP"]:
            return "M"  # Modifier
        elif token.dep_ in ["dobj", "pobj", "obj"]:
            return "O"  # Object
    return "M"  # Default to Modifier

# Step 2: Analyze sentences and extract SVO(M) structures
def analyze_sentence(sentence):
    doc = nlp(sentence)
    svo_structure = {"S": [], "V": [], "O": [], "M": []}
    sentiment_scores = {"S": 0, "V": 0, "O": 0, "M": 0}

    # Tokenize and label
    for token in doc:
        label = label_word(token.text)
        svo_structure[label].append(token.text)

    # Calculate sentiment for each part
    for part in ["S", "V", "O", "M"]:
        text = " ".join(svo_structure[part])
        if text:  # Only calculate if part exists
            sentiment_scores[part] = analyzer.polarity_scores(text)["compound"]

    # Determine structure weight
    detected_parts = [part for part in ["S", "V", "O", "M"] if svo_structure[part]]
    weight = len(detected_parts)

    # Calculate overall sentiment
    overall_sentiment = sum(sentiment_scores[part] for part in detected_parts) / weight if weight else 0
    return overall_sentiment, sentiment_scores, detected_parts

# Step 3: Analyze all sentences
results = {}
for sentence in sentences:
    overall_sentiment, sentiment_scores, detected_parts = analyze_sentence(sentence)
    structure = "".join(detected_parts)  # e.g., SVO
    results[sentence] = {
        "Overall Sentiment": round(overall_sentiment, 2),
        "Part Sentiments": sentiment_scores,
        "Detected Structure": structure,
        "Percentage Sentiment": round(overall_sentiment * 100, 2),
    }

# Display results
for sentence, result in results.items():
    print(f"Sentence: {sentence}")
    print(f"Detected Structure: {result['Detected Structure']}")
    print(f"Part Sentiments: {result['Part Sentiments']}")
    print(f"Overall Sentiment: {result['Overall Sentiment']}")
    print(f"Percentage Sentiment: {result['Percentage Sentiment']}%")
    print("-" * 50)


Sentence: I love programming.
Detected Structure: SM
Part Sentiments: {'S': 0.6369, 'V': 0, 'O': 0, 'M': 0.0}
Overall Sentiment: 0.32
Percentage Sentiment: 31.85%
--------------------------------------------------
Sentence: I hate bugs.
Detected Structure: SVM
Part Sentiments: {'S': 0.0, 'V': -0.5719, 'O': 0, 'M': 0.0}
Overall Sentiment: -0.19
Percentage Sentiment: -19.06%
--------------------------------------------------
Sentence: Coding is fun.
Detected Structure: SVM
Part Sentiments: {'S': 0.5106, 'V': 0.0, 'O': 0, 'M': 0.0}
Overall Sentiment: 0.17
Percentage Sentiment: 17.02%
--------------------------------------------------
Sentence: Debugging is frustrating.
Detected Structure: VM
Part Sentiments: {'S': 0, 'V': 0.0, 'O': 0, 'M': -0.4404}
Overall Sentiment: -0.22
Percentage Sentiment: -22.02%
--------------------------------------------------
Sentence: She enjoys solving complex problems.
Detected Structure: SVM
Part Sentiments: {'S': -0.4019, 'V': 0.6908, 'O': 0, 'M': 0.0}
Over

In [10]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import spacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Example sentences
sentences = [
    "I love programming.",
    "I absolutely hate bugs so very much.",
    "Coding is fun.",
    "Debugging is frustrating.",
    "She enjoys solving complex problems.",
    "The movie was incredibly exciting but a bit long.",
]

# Step 1: Label words as S, V, O, or M
def label_word(word):
    doc = nlp(word)
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN", "PRON"]:
            return "S"  # Subject
        elif token.pos_ in ["VERB", "AUX"]:
            return "V"  # Verb
        elif token.pos_ in ["ADJ", "ADV", "DET", "ADP"]:
            return "M"  # Modifier
        elif token.dep_ in ["dobj", "pobj", "obj"]:
            return "O"  # Object
    return "M"  # Default to Modifier

# Step 2: Analyze sentences and extract SVO(M) structures
def analyze_sentence(sentence):
    doc = nlp(sentence)
    svo_structure = {"S": [], "V": [], "O": [], "M": []}
    sentiment_scores = {"S": 0, "V": 0, "O": 0, "M": 0}

    # Tokenize and label
    for token in doc:
        label = label_word(token.text)
        svo_structure[label].append(token.text)

    # Calculate sentiment for each part
    for part in ["S", "V", "O", "M"]:
        text = " ".join(svo_structure[part])
        if text:  # Only calculate if part exists
            sentiment_scores[part] = analyzer.polarity_scores(text)["compound"]

    # Determine structure weight
    detected_parts = [part for part in ["S", "V", "O", "M"] if svo_structure[part]]
    weight = len(detected_parts)

    # Calculate overall sentiment
    overall_sentiment = sum(sentiment_scores[part] for part in detected_parts) / weight if weight else 0
    return overall_sentiment, sentiment_scores, detected_parts

# Step 3: Analyze all sentences and append those with valid sentiments
results = {}
final_sentences = []  # To store sentences that meet the condition

for sentence in sentences:
    overall_sentiment, sentiment_scores, detected_parts = analyze_sentence(sentence)
    structure = "".join(detected_parts)  # e.g., SVO
    overall_percent_sentiment = round(overall_sentiment * 100, 2)

    # Check sentiment conditions (greater than 45% or less than -45%)
    if overall_percent_sentiment > 25 or overall_percent_sentiment < -15:
        final_sentences.append([sentence,"Percentage Sentiment: "+str(overall_percent_sentiment)])  # Append to result string

    results[sentence] = {
        "Overall Sentiment": round(overall_sentiment, 2),
        "Part Sentiments": sentiment_scores,
        "Detected Structure": structure,
        "Percentage Sentiment": overall_percent_sentiment,
    }

# Display selected sentences based on sentiment condition
print("Sentences with sentiment > 45% or < -45%:")
for sentence in final_sentences:
    print(f"- {sentence}")

# Optionally, display the full analysis
print("\nFull Analysis:")
for sentence, result in results.items():
    print(f"Sentence: {sentence}")
    print(f"Detected Structure: {result['Detected Structure']}")
    print(f"Part Sentiments: {result['Part Sentiments']}")
    print(f"Overall Sentiment: {result['Overall Sentiment']}")
    print(f"Percentage Sentiment: {result['Percentage Sentiment']}%")
    print("-" * 50)


Sentences with sentiment > 45% or < -45%:
- ['I love programming.', 'Percentage Sentiment: 31.85']
- ['I absolutely hate bugs so very much.', 'Percentage Sentiment: -19.06']
- ['Debugging is frustrating.', 'Percentage Sentiment: -22.02']

Full Analysis:
Sentence: I love programming.
Detected Structure: SM
Part Sentiments: {'S': 0.6369, 'V': 0, 'O': 0, 'M': 0.0}
Overall Sentiment: 0.32
Percentage Sentiment: 31.85%
--------------------------------------------------
Sentence: I absolutely hate bugs so very much.
Detected Structure: SVM
Part Sentiments: {'S': 0.0, 'V': -0.5719, 'O': 0, 'M': 0.0}
Overall Sentiment: -0.19
Percentage Sentiment: -19.06%
--------------------------------------------------
Sentence: Coding is fun.
Detected Structure: SVM
Part Sentiments: {'S': 0.5106, 'V': 0.0, 'O': 0, 'M': 0.0}
Overall Sentiment: 0.17
Percentage Sentiment: 17.02%
--------------------------------------------------
Sentence: Debugging is frustrating.
Detected Structure: VM
Part Sentiments: {'S': 0

In [17]:
from openai import AssistantEventHandler, OpenAI
c="".join([final_sentences[i][0].join(final_sentences[i][1]) for i in range(len(final_sentences))])
prompt="the information provided may give context on how the user interprets the text, your job is to pay careful attention to this and aid, advise the user on the highlighted contents shared here: "+c

print(prompt)
# Function to call the LLM API and get a response
client = OpenAI(api_key="sk-proj-vo15tZnQ5I6wqHkj0ip2T3BlbkFJZNs1KzScNjrZanX96TLw")
role="respond as an extremely knowledgable legal advisor in the south african business context"
response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "user", "content": role+" "+prompt}
                ]
            )
chat=response.choices[0].message.content
response_text = chat  # Replace with actual API response logic
response_text

the information provided may give context on how the user interprets the text, your job is to pay careful attention to this and aid, advise the user on the highlighted contents shared here: PI love programming.eI love programming.rI love programming.cI love programming.eI love programming.nI love programming.tI love programming.aI love programming.gI love programming.eI love programming. I love programming.SI love programming.eI love programming.nI love programming.tI love programming.iI love programming.mI love programming.eI love programming.nI love programming.tI love programming.:I love programming. I love programming.3I love programming.1I love programming..I love programming.8I love programming.5PI absolutely hate bugs so very much.eI absolutely hate bugs so very much.rI absolutely hate bugs so very much.cI absolutely hate bugs so very much.eI absolutely hate bugs so very much.nI absolutely hate bugs so very much.tI absolutely hate bugs so very much.aI absolutely hate bugs so ver

'As a highly experienced legal advisor in the South African business context, I am here to provide guidance on the contents you have shared. It appears that you have expressed a strong passion for programming, as well as a deep frustration with debugging. I understand that dealing with bugs can be a challenging and time-consuming task, but it is an important part of the programming process to ensure the functionality and efficiency of your code.\n\nIn the South African business context, it is crucial to prioritize clear communication and effective problem-solving skills in dealing with technical issues like debugging. It may be helpful to establish a systematic approach to debugging, such as using debugging tools, writing clear and concise code, and seeking help from others when necessary.\n\nAdditionally, it is important to maintain a positive attitude and persistence when facing challenges in programming. Remember that each debugging experience is an opportunity to learn and improve 

In [18]:
# prompt: use pywhatkit to send a message

!pip install pywhatkit

import pywhatkit

# Replace with your phone number and message
phone_number = "+27843954539"  # Replace with the recipient's phone number
message = response_text  # Replace with the message you want to send

# Send the message
try:
    pywhatkit.sendwhatmsg(phone_number, message, 20, 46) # Send at 2:45 PM
    print("Message sent successfully!")
except Exception as e:
    print(f"Error sending message: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.2/171.2 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for pyautogui: filename=PyAutoGUI-0.9.54-py3-none-any.whl size=37637 sha256=533a01b16bee

KeyError: 'DISPLAY'